In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import time
import math
from shapely.geometry import LineString, Point
from pyproj import CRS, Transformer

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "processed"
TRAIN_DIR = DATA_DIR / "train"
STUDENT_TEST_DIR = DATA_DIR / "student_test"
CANDIDATE_DIR = PROJECT_ROOT / "data" / "interim" / "candidates"
ALIGNED_DIR = PROJECT_ROOT / "data" / "interim" / "aligned"
CORRECTED_DIR = DATA_DIR / "corrected_trips_33_54"
OUTPUT_DIR = DATA_DIR / "features"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for p in [TRAIN_DIR, STUDENT_TEST_DIR, CANDIDATE_DIR / "train", CANDIDATE_DIR / "student_test"]:
    assert p.exists(), f"Missing directory: {p}"

print("Project:", PROJECT_ROOT)
print("Train candidates:", CANDIDATE_DIR / "train")
print("Student-test candidates:", CANDIDATE_DIR / "student_test")
print("Aligned labels:", ALIGNED_DIR)
print("Corrected labels:", CORRECTED_DIR)
print("Output:", OUTPUT_DIR)

Project: c:\Users\HP\.vscode\python\ml\mlcodes\ml_project\PathSnap
Train candidates: c:\Users\HP\.vscode\python\ml\mlcodes\ml_project\PathSnap\data\interim\candidates\train
Student-test candidates: c:\Users\HP\.vscode\python\ml\mlcodes\ml_project\PathSnap\data\interim\candidates\student_test
Aligned labels: c:\Users\HP\.vscode\python\ml\mlcodes\ml_project\PathSnap\data\interim\aligned
Corrected labels: c:\Users\HP\.vscode\python\ml\mlcodes\ml_project\PathSnap\data\processed\corrected_trips_33_54
Output: c:\Users\HP\.vscode\python\ml\mlcodes\ml_project\PathSnap\data\processed\features


In [2]:
def load_candidates(partition):
    folder = CANDIDATE_DIR / partition
    files = sorted(folder.glob("trip_*_candidates.csv"))
    assert files, f"No candidate CSVs found in {folder}"
    frames = []
    for file in files:
        df = pd.read_csv(file)
        if "trip_id" not in df.columns:
            df["trip_id"] = file.name.replace("trip_", "").replace("_candidates.csv", "")
        frames.append(df)
    result = pd.concat(frames, ignore_index=True)
    required = {"trip_id", "observation_pos", "arc_id", "candidate_rank", "distance_m"}
    missing = required - set(result.columns)
    assert not missing, f"{partition} candidates missing columns: {missing}"
    result["trip_id"] = result["trip_id"].astype(str).str.zfill(8)
    result["observation_pos"] = result["observation_pos"].astype(int)
    result["arc_id"] = result["arc_id"].astype(int)
    return result

train_candidates = load_candidates("train")
student_test_candidates = load_candidates("student_test")

print("Training candidate rows:", len(train_candidates))
print("Training trips:", train_candidates["trip_id"].nunique())
print("Student-test candidate rows:", len(student_test_candidates))
print("Student-test trips:", student_test_candidates["trip_id"].nunique())
display(train_candidates.head())

Training candidate rows: 2607266
Training trips: 80
Student-test candidate rows: 275425
Student-test trips: 10


,trip_id,observation_pos,observation_id,arc_id,candidate_rank,distance_m,distance_to_projected_point_m,segment_length_m,gps_heading_deg,road_heading_deg,heading_difference_deg,projection_fraction,used_fallback,search_radius_m
0,00000001,0,0.0,64973,1,0.723049,0.723049,293.352246,8.630643,7.317066,1.313576,0.109775,False,100.0
1,00000001,0,0.0,117639,2,15.138380,15.138380,1020.835864,8.630643,187.386430,178.755787,0.512692,False,100.0
2,00000001,0,0.0,2641,3,32.210996,32.210996,425.215971,8.630643,7.188148,1.442495,1.000000,False,100.0
3,00000001,0,0.0,53932,4,93.046268,93.046268,169.107907,8.630643,356.786956,11.843687,0.154792,False,100.0
4,00000001,0,0.0,50652,5,93.046268,93.046268,169.107907,8.630643,176.786956,168.156313,0.845208,False,100.0


In [3]:
def read_training_labels():
    labels = []
    review_trips = {"00000033", "00000054"}

    for trip_id in sorted(train_candidates["trip_id"].unique()):
        if trip_id in review_trips:
            file = CORRECTED_DIR / f"{trip_id}_corrected_observations.csv"
            assert file.exists(), f"Missing corrected labels: {file}"
            df = pd.read_csv(file)
            if "observation_pos" in df.columns:
                df["observation_pos"] = df["observation_pos"].astype(int)
            elif "observation_id" in df.columns:
                df["observation_pos"] = pd.to_numeric(df["observation_id"], errors="raise").astype(int)
            else:
                raise ValueError(f"{trip_id}: corrected file has no observation_pos or observation_id")

            df["label_arc_id"] = pd.to_numeric(df["corrected_arc_id"], errors="raise").astype(int)
            df["label_source"] = "corrected_graph_nearest"
            df["label_confidence"] = df["match_confidence"].astype(str).str.lower()
            df["label_usable"] = df["label_confidence"].isin(["high", "medium"])
            labels.append(df[["observation_pos", "label_arc_id", "label_source", "label_confidence", "label_usable"]].assign(trip_id=trip_id))
        else:
            file = ALIGNED_DIR / f"{trip_id}.csv"
            assert file.exists(), f"Missing aligned labels for {trip_id}: {file}"
            df = pd.read_csv(file)
            df["observation_pos"] = pd.to_numeric(df["observation_pos"], errors="raise").astype(int)
            df["label_arc_id"] = pd.to_numeric(df["aligned_arc_id"], errors="raise").astype(int)
            df["label_source"] = "reference_route_alignment"
            df["label_confidence"] = "provisional"
            df["label_usable"] = True
            labels.append(df[["observation_pos", "label_arc_id", "label_source", "label_confidence", "label_usable"]].assign(trip_id=trip_id))

    result = pd.concat(labels, ignore_index=True)
    result["trip_id"] = result["trip_id"].astype(str).str.zfill(8)
    return result

train_labels = read_training_labels()
print("Label rows:", len(train_labels))
print("Labelled trips:", train_labels["trip_id"].nunique())
print("Usable observations:", int(train_labels["label_usable"].sum()))
display(train_labels.groupby(["trip_id", "label_source", "label_usable"]).size().reset_index(name="observations").head(20))

Label rows: 208417
Labelled trips: 80
Usable observations: 207311


,trip_id,label_source,label_usable,observations
0,00000001,reference_route_alignment,True,1315
1,00000002,reference_route_alignment,True,7243
2,00000003,reference_route_alignment,True,821
3,00000004,reference_route_alignment,True,1064
4,00000005,reference_route_alignment,True,1045
5,00000006,reference_route_alignment,True,1450
6,00000008,reference_route_alignment,True,3093
7,00000010,reference_route_alignment,True,2711
8,00000011,reference_route_alignment,True,2828
9,00000012,reference_route_alignment,True,1608


In [4]:
def make_labelled_candidates(candidates, labels):
    data = candidates.merge(
        labels,
        on=["trip_id", "observation_pos"],
        how="inner",
        validate="many_to_one"
    )
    data = data[data["label_usable"]].copy()
    data["target"] = (data["arc_id"] == data["label_arc_id"]).astype(int)

    coverage = data.groupby(["trip_id", "observation_pos"]).agg(
        target_found=("target", "max"),
        candidate_count=("arc_id", "size")
    ).reset_index()
    coverage["candidate_miss"] = 1 - coverage["target_found"]
    return data, coverage

train_data, train_coverage = make_labelled_candidates(train_candidates, train_labels)

print("Training rows:", len(train_data))
print("Positive candidate rows:", int(train_data["target"].sum()))
print("Negative candidate rows:", int((train_data["target"] == 0).sum()))
print("Usable observations:", len(train_coverage))
print("Candidate misses:", int(train_coverage["candidate_miss"].sum()))
print("Candidate miss rate:", round(train_coverage["candidate_miss"].mean() * 100, 3), "%")
display(train_coverage.groupby("trip_id")["candidate_miss"].mean().sort_values(ascending=False).head(10))

Training rows: 2606160
Positive candidate rows: 206042
Negative candidate rows: 2400118
Usable observations: 207311
Candidate misses: 1269
Candidate miss rate: 0.612 %


trip_id
00000002    0.103824
00000066    0.045780
00000072    0.040248
00000043    0.034884
00000093    0.024773
00000061    0.024476
00000081    0.021712
00000019    0.011895
00000020    0.007650
00000028    0.003366
Name: candidate_miss, dtype: float64

In [5]:
def make_transformer(track):
    lon0 = float(track["longitude"].median())
    lat0 = float(track["latitude"].median())
    crs = CRS.from_proj4(
        f"+proj=aeqd +lat_0={lat0} +lon_0={lon0} +datum=WGS84 +units=m +no_defs"
    )
    return Transformer.from_crs("EPSG:4326", crs, always_xy=True)

def load_trip(folder):
    return {
        "track": pd.read_csv(folder / "track.csv"),
        "nodes": pd.read_csv(folder / "nodes.csv"),
        "arcs": pd.read_csv(folder / "arcs.csv"),
        "route": pd.read_csv(folder / "route.csv")
    }

def get_route_arc_ids(route):
    possible = ["arc_id", "arc_ids", "route_arc_id"]
    col = next((c for c in possible if c in route.columns), None)
    if col is None:
        if route.shape[1] == 1:
            col = route.columns[0]
        else:
            raise ValueError(f"Cannot identify arc ID column in route.csv: {route.columns.tolist()}")
    return pd.to_numeric(route[col], errors="raise").astype(int).to_numpy()

def align_student_trip(trip):
    track, nodes, arcs = trip["track"], trip["nodes"], trip["arcs"]
    transformer = make_transformer(track)
    gx, gy = transformer.transform(track["longitude"].to_numpy(float), track["latitude"].to_numpy(float))
    nx, ny = transformer.transform(nodes["longitude"].to_numpy(float), nodes["latitude"].to_numpy(float))

    route_arc_ids = get_route_arc_ids(trip["route"])
    src = pd.to_numeric(arcs["from_node_id"], errors="raise").astype(int).to_numpy()
    dst = pd.to_numeric(arcs["to_node_id"], errors="raise").astype(int).to_numpy()

    route_geoms = []
    valid_ids = []
    for arc_id in route_arc_ids:
        if arc_id < 0 or arc_id >= len(arcs):
            continue
        u, v = src[arc_id], dst[arc_id]
        if u < 0 or v < 0 or u >= len(nodes) or v >= len(nodes):
            continue
        geom = LineString([(nx[u], ny[u]), (nx[v], ny[v])])
        if geom.length > 0:
            route_geoms.append(geom)
            valid_ids.append(int(arc_id))

    if not route_geoms:
        raise ValueError("No usable route geometries")

    n, m = len(track), len(route_geoms)
    distances = np.empty((n, m), dtype=np.float32)
    for j, geom in enumerate(route_geoms):
        distances[:, j] = [Point(float(x), float(y)).distance(geom) for x, y in zip(gx, gy)]

    dp = np.empty((n, m), dtype=np.float64)
    back = np.zeros((n, m), dtype=np.int32)
    dp[0] = distances[0] ** 2
    for i in range(1, n):
        best = np.minimum.accumulate(dp[i - 1])
        best_idx = np.empty(m, dtype=np.int32)
        current = 0
        for j in range(m):
            if dp[i - 1, j] < dp[i - 1, current]:
                current = j
            best_idx[j] = current
        dp[i] = distances[i] ** 2 + best
        back[i] = best_idx

    steps = np.empty(n, dtype=np.int32)
    steps[-1] = int(np.argmin(dp[-1]))
    for i in range(n - 1, 0, -1):
        steps[i - 1] = back[i, steps[i]]

    return pd.DataFrame({
        "trip_id": "",
        "observation_pos": np.arange(n),
        "label_arc_id": np.asarray(valid_ids, dtype=int)[steps],
        "alignment_distance_m": distances[np.arange(n), steps]
    })

student_test_labels = []
student_test_folders = sorted(p for p in STUDENT_TEST_DIR.iterdir() if p.is_dir())
for folder in student_test_folders:
    trip = load_trip(folder)
    labels = align_student_trip(trip)
    labels["trip_id"] = folder.name.zfill(8)
    labels["label_source"] = "provisional_student_test_route_alignment"
    student_test_labels.append(labels)

student_test_labels = pd.concat(student_test_labels, ignore_index=True)
student_test_data = student_test_candidates.merge(
    student_test_labels,
    on=["trip_id", "observation_pos"],
    how="left",
    validate="many_to_one"
)
student_test_data["target"] = (
    student_test_data["arc_id"] == student_test_data["label_arc_id"]
).astype(int)

student_test_coverage = student_test_data.groupby(["trip_id", "observation_pos"]).agg(
    target_found=("target", "max"),
    candidate_count=("arc_id", "size"),
    alignment_distance_m=("alignment_distance_m", "first")
).reset_index()
student_test_coverage["candidate_miss"] = 1 - student_test_coverage["target_found"]

print("Student-test labelled observations:", len(student_test_coverage))
print("Student-test candidate miss rate:", round(student_test_coverage["candidate_miss"].mean() * 100, 3), "%")
print("Median route alignment distance:", round(student_test_coverage["alignment_distance_m"].median(), 2), "m")
display(student_test_coverage.groupby("trip_id").agg(
    observations=("observation_pos", "count"),
    miss_rate=("candidate_miss", "mean"),
    median_alignment_m=("alignment_distance_m", "median")
))

Student-test labelled observations: 19160
Student-test candidate miss rate: 0.099 %
Median route alignment distance: 5.27 m


,observations,miss_rate,median_alignment_m
trip_id,,,
00000000,2503,0.000400,6.049407
00000007,1354,0.000000,4.515841
00000009,1049,0.000000,4.706960
00000040,2995,0.000000,4.925128
00000052,262,0.000000,4.176026
00000056,3958,0.004548,4.794481
00000065,768,0.000000,4.400685
00000076,1486,0.000000,6.849242
00000077,3911,0.000000,6.117383


In [6]:
FEATURE_COLUMNS = [
    "candidate_rank",
    "distance_m",
    "distance_to_projected_point_m",
    "segment_length_m",
    "gps_heading_deg",
    "road_heading_deg",
    "heading_difference_deg",
    "projection_fraction",
    "used_fallback",
    "search_radius_m"
]
FEATURE_COLUMNS = [c for c in FEATURE_COLUMNS if c in train_data.columns and c in student_test_data.columns]
assert len(FEATURE_COLUMNS) >= 3, f"Too few common feature columns: {FEATURE_COLUMNS}"

for df in [train_data, student_test_data]:
    for col in FEATURE_COLUMNS:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df[FEATURE_COLUMNS] = df[FEATURE_COLUMNS].replace([np.inf, -np.inf], np.nan)
    df[FEATURE_COLUMNS] = df[FEATURE_COLUMNS].fillna(df[FEATURE_COLUMNS].median())

print("Features:", FEATURE_COLUMNS)
display(train_data[FEATURE_COLUMNS + ["target"]].head())

Features: ['candidate_rank', 'distance_m', 'distance_to_projected_point_m', 'segment_length_m', 'gps_heading_deg', 'road_heading_deg', 'heading_difference_deg', 'projection_fraction', 'used_fallback', 'search_radius_m']


,candidate_rank,distance_m,distance_to_projected_point_m,segment_length_m,gps_heading_deg,road_heading_deg,heading_difference_deg,projection_fraction,used_fallback,search_radius_m,target
0,1,0.723049,0.723049,293.352246,8.630643,7.317066,1.313576,0.109775,False,100.0,1
1,2,15.138380,15.138380,1020.835864,8.630643,187.386430,178.755787,0.512692,False,100.0,0
2,3,32.210996,32.210996,425.215971,8.630643,7.188148,1.442495,1.000000,False,100.0,0
3,4,93.046268,93.046268,169.107907,8.630643,356.786956,11.843687,0.154792,False,100.0,0
4,5,93.046268,93.046268,169.107907,8.630643,176.786956,168.156313,0.845208,False,100.0,0


In [7]:
train_output = OUTPUT_DIR / "train_candidates.csv"
student_output = OUTPUT_DIR / "student_test_candidates.csv"
coverage_output = OUTPUT_DIR / "train_candidate_coverage.csv"
student_coverage_output = OUTPUT_DIR / "student_test_candidate_coverage.csv"
labels_output = OUTPUT_DIR / "student_test_provisional_labels.csv"

train_data.to_csv(train_output, index=False)
student_test_data.to_csv(student_output, index=False)
train_coverage.to_csv(coverage_output, index=False)
student_test_coverage.to_csv(student_coverage_output, index=False)
student_test_labels.to_csv(labels_output, index=False)

feature_info = pd.DataFrame({
    "feature": FEATURE_COLUMNS,
    "dtype": [str(train_data[c].dtype) for c in FEATURE_COLUMNS],
    "missing_train": [int(train_data[c].isna().sum()) for c in FEATURE_COLUMNS],
    "missing_student_test": [int(student_test_data[c].isna().sum()) for c in FEATURE_COLUMNS]
})
feature_info.to_csv(OUTPUT_DIR / "feature_columns.csv", index=False)

print("Saved:")
for p in [train_output, student_output, coverage_output, student_coverage_output, labels_output, OUTPUT_DIR / "feature_columns.csv"]:
    print(p)

Saved:
c:\Users\HP\.vscode\python\ml\mlcodes\ml_project\PathSnap\data\processed\features\train_candidates.csv
c:\Users\HP\.vscode\python\ml\mlcodes\ml_project\PathSnap\data\processed\features\student_test_candidates.csv
c:\Users\HP\.vscode\python\ml\mlcodes\ml_project\PathSnap\data\processed\features\train_candidate_coverage.csv
c:\Users\HP\.vscode\python\ml\mlcodes\ml_project\PathSnap\data\processed\features\student_test_candidate_coverage.csv
c:\Users\HP\.vscode\python\ml\mlcodes\ml_project\PathSnap\data\processed\features\student_test_provisional_labels.csv
c:\Users\HP\.vscode\python\ml\mlcodes\ml_project\PathSnap\data\processed\features\feature_columns.csv


## 7. Final checks

Before model training, verify:
- All expected training trips are represented.
- Student-test trips are represented.
- Every training observation has a usable label.
- Candidate misses are reported, not silently corrected.
- Trips 33 and 54 use corrected labels, with low-confidence observations excluded.
- Student-test labels are kept separate from training data.

**Next notebook:** `07_model_training_and_comparison.ipynb`, which will train and compare at least five model families using trip-grouped validation.